# Cyberlindnera Fabianii

In [9]:
from cobra.io import read_sbml_model
from IPython.display import display
import pandas as pd

model_path = "models/Cyberlindnera_fabianii.sbml"
model = read_sbml_model(model_path)
solution = model.optimize()

print(f"Model ID: {model.id}")
print(f"Name: {model.name}")
print(f"Compartments: {len(model.compartments)}")
print(f"Metabolites: {len(model.metabolites)}")
print(f"Reactions: {len(model.reactions)}")
print(f"Exchange reactions: {len(model.exchanges)}")
print(f"Genes: {len(model.genes)}")
print(f"Objective: {model.objective.expression}")
print(f"Status: {solution.status}")
print(f"Objective value: {solution.objective_value}")


Model ID: Cyberlindnera_fabianii
Name: None
Compartments: 7
Metabolites: 1786
Reactions: 2453
Exchange reactions: 318
Genes: 1226
Objective: 1.0*BIOMASS_C - 1.0*BIOMASS_C_reverse_e6e22
Status: optimal
Objective value: 0.0


In [10]:
sol0 = model.optimize()
print("before:", sol0.status, sol0.objective_value)

med = model.medium.copy()
med["UF02997_E"] = 10.0   # biotin uptake
model.medium = med

sol1 = model.optimize()
print("after:", sol1.status, sol1.objective_value)
print(model.summary(solution=sol1))

before: optimal 0.0
after: optimal 1.1024183402449756
Objective
1.0 BIOMASS_C = 1.1024183402449756

Uptake
------
Metabolite  Reaction      Flux  C-Number  C-Flux
  ash_1g_e UF02549_E   0.05292         0   0.00%
      pi_e UF02765_E    0.2199         0   0.00%
     btn_e UF02997_E 0.0001102        10   0.00%
     fe2_e UF03268_E 0.0002205         0   0.00%
    glcD_e UF03288_E        10         6 100.00%
     nh4_e UF03376_E     6.227         0   0.00%
      o2_e UF03382_E     18.89         0   0.00%
     so4_e UF03456_E   0.06218         0   0.00%
       h_e UF03474_E     65.41         0   0.00%

Secretion
---------
  Metabolite  Reaction       Flux  C-Number C-Flux
       co2_e UF03227_E     -19.28         1 94.06%
biomass_1g_c UF03247_C     -1.102         0  0.00%
     gcald_e UF03286_E -0.0001102         2  0.00%
       h2o_e UF03314_E     -35.93         0  0.00%
     acglu_e UF03533_E    -0.1738         7  5.93%
        co_e UF04432_E -0.0001102         1  0.00%
    clpnbb_c UF046

In [11]:
n_reactions = 10
df = pd.DataFrame(
    [{
        "id": rxn.id,
        "name": rxn.name,
        "equation": rxn.build_reaction_string(use_metabolite_names=True),
        "lower_bound": rxn.lower_bound,
        "upper_bound": rxn.upper_bound
    } for rxn in model.reactions[:n_reactions]]
)
df

,id,name,equation,lower_bound,upper_bound
0,UF00014_C,primary alcohol:NAD+ oxidoreductase (1.1.1.1)_...,2-Methylbutanal + H+ + NADH(2-) --> 2-methylbu...,0.0,1000.0
1,UF00014_M,primary alcohol:NAD+ oxidoreductase (1.1.1.1)_...,2-Methylbutanal + H+ + NADH(2-) --> 2-methylbu...,0.0,1000.0
2,UF00015_C,primary alcohol:NAD+ oxidoreductase (1.1.1.1)_...,2-Methylbutanal + H+ + NADPH(4-) --> 2-methylb...,0.0,1000.0
3,UF00015_M,primary alcohol:NAD+ oxidoreductase (1.1.1.1)_...,2-Methylbutanal + H+ + NADPH(4-) --> 2-methylb...,0.0,1000.0
4,UF00019_C,choline dehydrogenase (1.1.1.1)_1_1_1_1_1_1_1_...,choline + NAD(1-) --> betaine aldehyde + H+ + ...,0.0,1000.0
5,UF00020_C,choline dehydrogenase (1.1.1.1)_1_1_1_1_1_1_1_...,choline + NADP(3-) --> betaine aldehyde + H+ +...,0.0,1000.0
6,UF00022_C,ethanol:NAD+ oxidoreductase (1.1.1.1|1.1.1.71)...,acetaldehyde + H+ + NADH(2-) <=> ethanol + NAD...,-1000.0,1000.0
7,UF00023_C,1-Octanol:NAD+ oxidoreductase (1.1.1.1|1.1.1.7...,H+ + NADH(2-) + octanal <=> NAD(1-) + octan-1-ol,-1000.0,1000.0
8,UF00023_M,1-Octanol:NAD+ oxidoreductase (1.1.1.1|1.1.1.7...,H+ + NADH(2-) + octanal <=> NAD(1-) + octan-1-ol,-1000.0,1000.0
9,UF00026_C,3-oxoacyl-[acyl-carrier-protein] reductase (1....,3-oxo-decanoyl-[acp] + H+ + NADPH(4-) --> (R)-...,0.0,1000.0


In [12]:
n_exchanges = 30
active_solution = sol1 if "sol1" in locals() else model.optimize()

exchange_df = pd.DataFrame(
    [{
        "id": ex.id,
        "name": ex.name,
        "metabolite_id": (next(iter(ex.metabolites)).id if ex.metabolites else None),
        "metabolite_name": (next(iter(ex.metabolites)).name if ex.metabolites else None),
        "lower_bound": ex.lower_bound,
        "upper_bound": ex.upper_bound,
        "flux": float(active_solution.fluxes.get(ex.id, 0.0)),
    } for ex in model.exchanges]
)

exchange_df.head(n_exchanges)

,id,name,metabolite_id,metabolite_name,lower_bound,upper_bound,flux
0,UF02481_E,(2-hydroxyphenyl)acetate exchange_1_1_1_1_1_1_...,2hyoxplac_e,(2-hydroxyphenyl)acetate,-0.0,1000.0,0.000000
1,UF02483_E,2-methylbutyrate exchange_1_1_1_1_1_1_1_1_1_1_...,2mba_e,2-methylbutyrate,-0.0,1000.0,0.000000
2,UF02500_E,3-Methylbutanoate exchange_1_1_1_1_1_1_1_1_1_1...,3mb_e,3-methylbutanoate,-0.0,1000.0,0.000000
3,UF02510_E,(4-Hydroxyphenyl)acetate exchange_1_1_1_1_1_1_...,4hoxpac_e,(4-hydroxyphenyl)acetate,-0.0,1000.0,0.000000
4,UF02516_E,8-amino-7-oxononanoic acid zwitterion exchange...,8aonn_e,8-amino-7-oxonooic acid zwitterion,-0.0,1000.0,0.000000
5,UF02549_E,ash 1 g exchange_1_1_1_1_1_1_1_1_1_1_1_1_1_1_1...,ash_1g_e,ash 1 g,-1000.0,1000.0,-0.052916
6,UF02691_E,Inosine exchange_1_1_1_1_1_1_1_1_1_1_1_1_1_1_1...,ins_e,inosine,-0.0,1000.0,0.000000
7,UF02751_E,phenylacetate exchange_1_1_1_1_1_1_1_1_1_1_1_1...,pac_e,phenylacetate,-0.0,1000.0,0.000000
8,UF02765_E,hydrogenphosphate exchange_1_1_1_1_1_1_1_1_1_1...,pi_e,hydrogenphosphate,-1000.0,1000.0,-0.219933
9,UF02802_E,spermine(4 exchange_1_1_1_1_1_1_1_1_1_1_1_1_1_...,sprm_e,spermine(4+),-0.0,1000.0,0.000000
